In [2]:
import sqlite3
import pandas as pd
import csv, os, time
from datetime import datetime

# ---------- SDM-LOGGER ----------
LOG_SDM = 'Log_SDM.csv'
if not os.path.exists(LOG_SDM):
    with open(LOG_SDM, 'w', newline='', encoding='utf-8') as f:
        csv.writer(f).writerow([
            'timestamp', 'run_id', 'stap', 'tabel', 'status',
            'aantal_rijen', 'duur_sec', 'melding'
        ])

RUN_ID_SDM = datetime.now().strftime('%Y%m%d_%H%M%S')

def log_sdm(stap, tabel='', status='OK', aantal_rijen=0, duur_sec=0.0, melding=''):
    with open(LOG_SDM, 'a', newline='', encoding='utf-8') as f:
        csv.writer(f).writerow([
            datetime.now().isoformat(timespec='seconds'),
            RUN_ID_SDM, stap, tabel, status,
            int(aantal_rijen), round(float(duur_sec), 3), melding
        ])
# ---------- /SDM-LOGGER ----------

# 1. Bestandsnamen
sql_bestand = 'schema.sql'
db_bestand = 'biketodrive.db'

# 2. Verbind met SQLite (wordt aangemaakt als het nog niet bestaat)
conn = sqlite3.connect(db_bestand)
cursor = conn.cursor()
start = time.time()

try:
    # 3. Lees het SQL-script
    with open(sql_bestand, 'r') as file:
        sql_script = file.read()

    # 4. Voer het volledige SQL-script uit
    cursor.executescript(sql_script)
    conn.commit()
    print(f"✅ De database '{db_bestand}' is succesvol aangemaakt en het script is uitgevoerd.")

    # 5. Verifieer welke tabellen bestaan
    query = "SELECT name FROM sqlite_master WHERE type='table';"
    df_tabellen = pd.read_sql_query(query, conn)
    print("\nDe volgende tabellen zijn gevonden in de database:")
    print(df_tabellen)

    # 6. Log per tabel een regel met het aantal rijen (vulling vanuit de operationele bron)
    for tabel in df_tabellen['name']:
        try:
            n = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {tabel}", conn).iloc[0]['n']
        except Exception as e:
            n = 0
            log_sdm('COUNT', tabel=tabel, status='ERROR', melding=str(e))
            continue
        log_sdm('LOAD_TABLE', tabel=tabel, status='OK', aantal_rijen=int(n))

    # 7. Samenvattende regel
    log_sdm('SCHEMA_EXECUTE', status='OK',
            duur_sec=time.time() - start,
            melding=f"{len(df_tabellen)} tabellen in SDM")
    print(f"\n📝 SDM-log weggeschreven naar {LOG_SDM} (run_id={RUN_ID_SDM})")

except Exception as e:
    log_sdm('SCHEMA_EXECUTE', status='ERROR',
            duur_sec=time.time() - start, melding=str(e))
    print(f"❌ Er is een fout opgetreden: {e}")

finally:
    conn.close()

✅ De database 'biketodrive.db' is succesvol aangemaakt en het script is uitgevoerd.

De volgende tabellen zijn gevonden in de database:
                  name
0              Filiaal
1                Klant
2            Fabrikant
3          Leverancier
4              Monteur
5                Fiets
6           Accessoire
7        Fiets_Verkoop
8   Accessoire_Verkoop
9    Accessoire_Inkoop
10        Fiets_Inkoop
11           Onderhoud

📝 SDM-log weggeschreven naar Log_SDM.csv (run_id=20260408_143338)
